In [ ]:
!pip -q install -U transformers datasets accelerate peft evaluate jiwer soundfile librosa openai-whisper
!pip -q install sentencepiece protobuf hf_transfer
!pip -q install --no-deps unsloth_zoo bitsandbytes accelerate peft trl triton unsloth
!pip -q install transformers==4.56.2
!pip -q install --no-deps trl==0.22.2
!pip -q install snac torchcodec "datasets>=3.4.1,<4.0.0" torchaudio

In [ ]:
!pip -q install librosa soundfile jiwer editdataset ystoi pesq

In [ ]:
!apt-get -qq update && apt-get -qq install -y ffmpeg

In [ ]:
import torch
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import whisper, torchaudio
import pandas as pd
import numpy as np
import os, glob, torch, librosa
from datasets import Dataset
from transformers import WhisperProcessor, WhisperForConditionalGeneration, Seq2SeqTrainingArguments, Seq2SeqTrainer
from peft import LoraConfig, get_peft_model, TaskType, PeftModel
import evaluate
import numpy as np
from IPython.display import Audio, display

In [ ]:
#PRETRAINED
from transformers import WhisperProcessor, WhisperForConditionalGeneration
import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BASE_MODEL = "openai/whisper-small"

processor_pre = WhisperProcessor.from_pretrained(BASE_MODEL)

model_pre = WhisperForConditionalGeneration.from_pretrained(BASE_MODEL).to(DEVICE)

model_pre.config.forced_decoder_ids = processor_pre.get_decoder_prompt_ids(
    language="tl",
    task="transcribe"
)

model_pre.eval()

print("Pretrained model ready.")

LOAD MODELS

In [ ]:
######################
# WHISPER (FINAL CLEAN VERSION)
######################

import torch, librosa
from transformers import WhisperProcessor, WhisperForConditionalGeneration
from peft import PeftModel

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

BASE_MODEL = "openai/whisper-small"
ADAPTER_DIR = "/content/drive/MyDrive/whisper_tl_impaired_lora_adapter"

processor = WhisperProcessor.from_pretrained(BASE_MODEL)
base_model = WhisperForConditionalGeneration.from_pretrained(BASE_MODEL).to(DEVICE)

base_model.config.forced_decoder_ids = processor.get_decoder_prompt_ids(
    language="tl",
    task="transcribe")

model = PeftModel.from_pretrained(base_model, ADAPTER_DIR).to(DEVICE)
model.eval()

print("Whisper +LoRA successfully loaded")

Whisper +LoRA successfully loaded


In [ ]:
######################
# UNSLOTH
######################
from unsloth import FastLanguageModel
import torch

LORA_PATH = "/content/drive/MyDrive/orpheus_lora_tagalog"  # change if needed

# Load base model again
model2, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/orpheus-3b-0.1-ft",
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=False,
)

# Attach your LoRA adapters
model2.load_adapter(LORA_PATH)

# Switch to inference mode (important!)
FastLanguageModel.for_inference(model2)

model2 = model2.to("cuda")
print("Model + LoRA loaded successfully.")

from snac import SNAC

snac_model = SNAC.from_pretrained("hubertsiuzdak/snac_24khz")
snac_model = snac_model.to("cpu")  # decoding on CPU is fine



In [ ]:
import re

def normalize_tagalog_text(text):
    text = text.lower()

    # Remove multiple spaces
    text = re.sub(r"\s+", " ", text)

    # Remove weird symbols
    text = re.sub(r"[^a-zA-Z0-9ñÑáéíóúÁÉÍÓÚ\s]", "", text)

    # Optional: basic Tagalog fixes
    text = text.replace("  ", " ")

    return text.strip()

Inference

In [ ]:
from torch.cuda import temperature
from IPython.display import Audio
###################################

# 9 15 22 25 26 28 32 40 45 48 52 56 57 62 64

# File path here
SPEAKER = "speaker_05"
UTT_ID = 64
###################################

wav_path = f"/content/drive/MyDrive/REAL DATA/dataset/impaired/{SPEAKER}/{UTT_ID:02d}.wav"
txt_path = f"/content/drive/MyDrive/REAL DATA/dataset/transcripts/{UTT_ID:02d}.txt"

print("BASE IMP AUDIO:")
display(Audio(wav_path))
print("\nBASE TRANSCRIPT:")
print(open(txt_path, "r", encoding="utf-8").read().strip())


######################
# WHISPER
######################
import torch, librosa
from transformers import WhisperProcessor, WhisperForConditionalGeneration
from peft import PeftModel

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

BASE_MODEL = "openai/whisper-small"
ADAPTER_DIR = "/content/drive/MyDrive/whisper_tl_impaired_lora_adapter"

processor = WhisperProcessor.from_pretrained(ADAPTER_DIR)
base = WhisperForConditionalGeneration.from_pretrained(BASE_MODEL).to(DEVICE)
base.config.forced_decoder_ids = processor.get_decoder_prompt_ids(language="tl", task="transcribe")
model = PeftModel.from_pretrained(base, ADAPTER_DIR).to(DEVICE)
model.eval()

audio, sr = librosa.load(wav_path, sr=16000)
features = processor.feature_extractor(audio, sampling_rate=16000).input_features
features = torch.tensor(features).to(DEVICE)

with torch.no_grad():
    predicted_ids = model.generate(
        input_features=features,
        max_new_tokens=256,
        num_beams=5,
        length_penalty=1.0,
        early_stopping=True,
        no_repeat_ngram_size=3,
    )

raw_text = processor.tokenizer.decode(predicted_ids[0], skip_special_tokens=True)
text = normalize_tagalog_text(raw_text)

print("\nTRANSCRIPTION:")
print(text, end="\n")

######################
# UNSLOTH
######################
from IPython.display import Audio, display
import torch

def generate_speech(text, chosen_voice=None):

    if chosen_voice:
        text = f"speaker_09: {text}"

    start_token = torch.tensor([[128259]], dtype=torch.int64)
    end_tokens  = torch.tensor([[128009, 128260]], dtype=torch.int64)

    input_ids = tokenizer(text, return_tensors="pt").input_ids
    input_ids = torch.cat([start_token, input_ids, end_tokens], dim=1).to("cuda")

    attention_mask = torch.ones_like(input_ids)

    generated_ids = model2.generate(
        input_ids=input_ids,
        attention_mask=attention_mask,
        max_new_tokens=1200,
        do_sample=False,

        repetition_penalty=1.2,
        eos_token_id=128258,
        use_cache=True
    )

    token_to_find = 128257
    idx = (generated_ids == token_to_find).nonzero(as_tuple=True)

    if len(idx[1]) > 0:
        generated_ids = generated_ids[:, idx[1][-1].item()+1:]

    generated_ids = generated_ids[generated_ids != 128258]
    trimmed = generated_ids[: (generated_ids.shape[0] // 7) * 7]
    trimmed = [t.item() - 128266 for t in trimmed]

    layer_1, layer_2, layer_3 = [], [], []
    for i in range((len(trimmed)+1)//7):
        layer_1.append(trimmed[7*i])
        layer_2.append(trimmed[7*i+1]-4096)
        layer_3.append(trimmed[7*i+2]-(2*4096))
        layer_3.append(trimmed[7*i+3]-(3*4096))
        layer_2.append(trimmed[7*i+4]-(4*4096))
        layer_3.append(trimmed[7*i+5]-(5*4096))
        layer_3.append(trimmed[7*i+6]-(6*4096))

    codes = [
        torch.tensor(layer_1).unsqueeze(0),
        torch.tensor(layer_2).unsqueeze(0),
        torch.tensor(layer_3).unsqueeze(0),
    ]

    audio_hat = snac_model.decode(codes)
    return audio_hat.detach().squeeze().cpu().numpy()



In [ ]:
recon_np = generate_speech(text, chosen_voice="speaker_09")
display(Audio(recon_np, rate=24000))

In [ ]:
recon_np = generate_speech("ito ay speech reconstruction", chosen_voice="speaker_09")
display(Audio(recon_np, rate=24000))

In [ ]:
tempsave = recon_np

In [ ]:
#Save run
import os
import shutil
import soundfile as sf

SAVE_ROOT = "/content/drive/MyDrive/deymoow"
os.makedirs(SAVE_ROOT, exist_ok=True)

# Extract numeric speaker ID (e.g., speaker_01 → 01)
spk_num = SPEAKER.split("_")[1]
utt_str = f"{UTT_ID:02d}"

base_name = f"{spk_num}_{utt_str}"

shutil.copy2(wav_path, f"{SAVE_ROOT}/{base_name}_imp.wav")

ref_text = open(txt_path, "r", encoding="utf-8").read().strip()
with open(f"{SAVE_ROOT}/{base_name}_ts.txt", "w", encoding="utf-8") as f:
    f.write(ref_text)

with open(f"{SAVE_ROOT}/{base_name}_gen.txt", "w", encoding="utf-8") as f:
    f.write(text.strip())

sf.write(f"{SAVE_ROOT}/{base_name}_recon.wav", recon_np, 24000)

print(f"Saved files for {base_name} in {SAVE_ROOT}")


Saved files for 15_64 in /content/drive/MyDrive/deymoow


Evaluate

In [ ]:
!pip -q install jiwer editdistance pystoi pesq phonemizer

In [ ]:
import os, re, numpy as np, librosa, torch
from jiwer import wer
import editdistance
from pystoi import stoi
from pesq import pesq

In [ ]:
# -------- Paths --------
IMPAIRED_ROOT = "/content/drive/MyDrive/REAL DATA/dataset/impaired"
CLEAN_ROOT    = "/content/drive/MyDrive/REAL DATA/dataset/clear"
TXT_ROOT      = "/content/drive/MyDrive/REAL DATA/dataset/transcripts"

EVAL_SPEAKERS = [f"speaker_{i:02d}" for i in [13, 14, 15]]
UTT_IDS = range(1, 71)

In [ ]:
def whisper_transcribe_path(wav_path: str) -> str:
    audio, _ = librosa.load(wav_path, sr=16000)
    features = processor.feature_extractor(audio, sampling_rate=16000).input_features
    features = torch.tensor(features).to(DEVICE)

    with torch.no_grad():
        predicted_ids = model.generate(
            input_features=features,
            max_new_tokens=256,
            num_beams=5,
            length_penalty=1.0,
            early_stopping=True,
        )

    raw = processor.tokenizer.decode(predicted_ids[0], skip_special_tokens=True)
    return normalize_tagalog_text(raw)

import numpy as np

def generate_speech_audio_24k(text: str, chosen_voice=None) -> np.ndarray:

    text = f"speaker_01: {text}"

    start_token = torch.tensor([[128259]], dtype=torch.int64)
    end_tokens  = torch.tensor([[128009, 128260]], dtype=torch.int64)

    input_ids = tokenizer(text, return_tensors="pt").input_ids
    input_ids = torch.cat([start_token, input_ids, end_tokens], dim=1).to("cuda")
    attention_mask = torch.ones_like(input_ids)

    with torch.no_grad():
        generated_ids = model2.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=1200,
            do_sample=False,
            repetition_penalty=1.1,
            eos_token_id=128258,
            use_cache=True
        )

    token_to_find = 128257
    idx = (generated_ids == token_to_find).nonzero(as_tuple=True)
    if len(idx[1]) > 0:
        generated_ids = generated_ids[:, idx[1][-1].item()+1:]

    generated_ids = generated_ids[generated_ids != 128258]

    trimmed = generated_ids[: (generated_ids.shape[0] // 7) * 7]
    trimmed = [t.item() - 128266 for t in trimmed]

    layer_1, layer_2, layer_3 = [], [], []
    for i in range((len(trimmed)+1)//7):
        layer_1.append(trimmed[7*i])
        layer_2.append(trimmed[7*i+1]-4096)
        layer_3.append(trimmed[7*i+2]-(2*4096))
        layer_3.append(trimmed[7*i+3]-(3*4096))
        layer_2.append(trimmed[7*i+4]-(4*4096))
        layer_3.append(trimmed[7*i+5]-(5*4096))
        layer_3.append(trimmed[7*i+6]-(6*4096))

    codes = [
        torch.tensor(layer_1).unsqueeze(0),
        torch.tensor(layer_2).unsqueeze(0),
        torch.tensor(layer_3).unsqueeze(0),
    ]

    audio_hat = snac_model.decode(codes)  # torch tensor
    audio_hat_np = audio_hat.detach().squeeze().cpu().numpy().astype(np.float32)
    return audio_hat_np

def eval_pack(speaker: str, utt_id: int):
    imp_wav  = f"{IMPAIRED_ROOT}/{speaker}/{utt_id:02d}.wav"
    clean_wav = f"{CLEAN_ROOT}/{speaker}/{utt_id:02d}.wav"
    txt_path = f"{TXT_ROOT}/{utt_id:02d}.txt"

    if not (os.path.exists(imp_wav) and os.path.exists(clean_wav) and os.path.exists(txt_path)):
        return None  # let eval loop skip

    ref_text = open(txt_path, "r", encoding="utf-8").read().strip()
    hyp_text = whisper_transcribe_path(imp_wav)

    recon_audio_24k = generate_speech_audio_24k(hyp_text)

    return {
        "speaker": speaker,
        "utt": utt_id,
        "imp_wav": imp_wav,
        "clean_wav": clean_wav,
        "ref_text": ref_text,
        "hyp_text": hyp_text,
        "recon_audio_24k": recon_audio_24k,
    }


In [ ]:
#PRETRAINED
def whisper_transcribe_pretrained(wav_path):
    audio, _ = librosa.load(wav_path, sr=16000)

    features = processor_pre.feature_extractor(
        audio,
        sampling_rate=16000
    ).input_features

    features = torch.tensor(features).to(DEVICE)

    with torch.no_grad():
        pred_ids = model_pre.generate(
            input_features=features,
            max_new_tokens=256,
            num_beams=5
        )

    return processor_pre.tokenizer.decode(
        pred_ids[0],
        skip_special_tokens=True
    )


In [ ]:
#Finetuned
def whisper_transcribe_finetuned(wav_path):
    audio, _ = librosa.load(wav_path, sr=16000)

    features = processor_pre.feature_extractor(
        audio,
        sampling_rate=16000
    ).input_features

    features = torch.tensor(features).to(DEVICE)

    with torch.no_grad():
        pred_ids = model_ft.generate(
            input_features=features,
            max_new_tokens=256,
            num_beams=5
        )

    return processor_pre.tokenizer.decode(
        pred_ids[0],
        skip_special_tokens=True
    )

In [ ]:
# ============================
# BATCHED WHISPER + PROGRESS
# ============================

import time
import numpy as np

def whisper_transcribe_paths_batch(wav_paths, batch_size=8):
    outputs = []
    for i in range(0, len(wav_paths), batch_size):
        chunk = wav_paths[i:i+batch_size]

        audios = [librosa.load(p, sr=16000)[0] for p in chunk]

        feats = processor.feature_extractor(
            audios,
            sampling_rate=16000,
            return_tensors="pt"
        ).input_features.to(DEVICE)

        with torch.no_grad():
            pred_ids = model.generate(
                input_features=feats,
                max_new_tokens=256,
                num_beams=5,
                length_penalty=1.0,
                early_stopping=True,
            )

        texts = processor.tokenizer.batch_decode(
            pred_ids,
            skip_special_tokens=True
        )

        outputs.extend([normalize_tagalog_text(t) for t in texts])

    return outputs


In [ ]:
def norm_text(text):
    text = text.lower()
    text = text.strip()
    text = re.sub(r"\s+"," ",text)
    return text

In [ ]:
import editdistance

def per(ref, hyp):
    ref_chars = list(norm_text(ref))
    hyp_chars = list(norm_text(hyp))

    return editdistance.eval(ref_chars, hyp_chars) / max(1, len(ref_chars))

In [ ]:
# ============================
# EVAL LOOP (speakers 13–15)
# ============================

EVAL_SPEAKERS = [f"speaker_{i:02d}" for i in [13, 14, 15]]
UTT_IDS = range(1, 71)

results = []
skipped = 0

total = len(EVAL_SPEAKERS) * len(list(UTT_IDS))
done = 0

PRINT_EVERY = 5
BATCH_SIZE_WHISPER = 8  # reduce if OOM

t0 = time.time()

for spk in EVAL_SPEAKERS:

    items = []

    for utt in UTT_IDS:
        imp_wav   = f"{IMPAIRED_ROOT}/{spk}/{utt:02d}.wav"
        clean_wav = f"{CLEAN_ROOT}/{spk}/{utt:02d}.wav"
        txt_path  = f"{TXT_ROOT}/{utt:02d}.txt"

        if not (os.path.exists(imp_wav) and os.path.exists(clean_wav) and os.path.exists(txt_path)):
            skipped += 1
            continue

        ref_text = open(txt_path, "r", encoding="utf-8").read().strip()
        items.append((utt, imp_wav, clean_wav, ref_text))

    # ---- Batch Whisper (GPU speedup)
    hyp_texts = whisper_transcribe_paths_batch(
        [x[1] for x in items],
        batch_size=BATCH_SIZE_WHISPER
    )

    # ---- TTS + Metrics (sequential, safe)
    for (utt, imp_wav, clean_wav, ref_text), hyp_text in zip(items, hyp_texts):

        recon_audio_24k = generate_speech_audio_24k(hyp_text)

        wer_val = float(wer(norm_text(ref_text), norm_text(hyp_text)))
        per_val = float(per(ref_text, hyp_text))
        pesq_val, stoi_val = pesq_stoi(clean_wav, recon_audio_24k)

        results.append({
            "speaker": spk,
            "utt": utt,
            "wer": wer_val,
            "per": per_val,
            "pesq": pesq_val,
            "stoi": stoi_val,
        })

        done += 1

        if done % PRINT_EVERY == 0:
            dt = time.time() - t0
            it_s = done / max(1e-9, dt)
            eta_s = (total - done) / max(1e-9, it_s)

            print(
                f"[{done:>3}/{total}] {spk} utt {utt:02d} | "
                f"WER={wer_val:.3f} PER={per_val:.3f} "
                f"PESQ={pesq_val:.3f} STOI={stoi_val:.3f} | "
                f"{it_s:.2f} it/s | ETA {eta_s/60:.1f} min"
            )


In [ ]:
import pandas as pd

df = pd.DataFrame(results)

print("\n==============================")
print("Scored:", len(df), "| Skipped:", skipped)
print("==============================\n")

display(df.head())

if len(df) > 0:
    print("=== OVERALL (speakers 13–15) ===")
    print("WER :", df["wer"].mean())
    print("PER :", df["per"].mean())
    print("PESQ:", df["pesq"].mean())
    print("STOI:", df["stoi"].mean())

    print("\n=== BY SPEAKER ===")
    display(df.groupby("speaker")[["wer","per","pesq","stoi"]].mean())



Scored: 183 | Skipped: 27



,speaker,utt,wer,per,pesq,stoi
0,speaker_13,1,1.0,0.200000,1.114222,0.336695
1,speaker_13,2,0.0,0.000000,1.125087,-0.108059
2,speaker_13,3,1.0,0.600000,1.040766,0.361367
3,speaker_13,4,1.0,1.000000,1.135320,0.000010
4,speaker_13,5,1.0,0.666667,1.060468,0.288862


=== OVERALL (speakers 13–15) ===
WER : 0.4400728597449909
PER : 0.2524058102791081
PESQ: 1.1221191348925315
STOI: 0.16497508198630442

=== BY SPEAKER ===


,wer,per,pesq,stoi
speaker,,,,
speaker_13,0.693687,0.444262,1.103936,0.159227
speaker_14,0.297872,0.132877,1.150282,0.167751
speaker_15,0.296429,0.151767,1.120354,0.168531


In [ ]:
SAVE_EVAL = "/content/drive/MyDrive/eval_fromcopyver.csv"
df.to_csv(SAVE_EVAL, index=False)
print("saved eval to:", SAVE_EVAL)

saved eval to: /content/drive/MyDrive/eval_fromcopyver.csv
